# EA2 - Actividad 2.4: Machine Learning con Spark MLlib

## Objetivos
- Entender los conceptos fundamentales de Spark MLlib
- Construir un pipeline de Machine Learning completo
- Aplicar feature engineering con Transformers
- Entrenar modelos de clasificacion (DecisionTree, RandomForest)
- Evaluar el rendimiento de los modelos

## Conceptos Clave

### Spark MLlib

MLlib es la libreria de Machine Learning de Spark. Permite entrenar modelos sobre datos distribuidos, escalando a millones de registros. Usa la API de DataFrames (`spark.ml`).

### Componentes Principales

| Componente | Descripcion | Ejemplo |
|------------|-------------|--------|
| **Transformer** | Transforma un DataFrame en otro | `StringIndexer`, `VectorAssembler` |
| **Estimator** | Algoritmo que se entrena sobre datos | `DecisionTreeClassifier`, `RandomForestClassifier` |
| **Pipeline** | Cadena de Transformers + Estimators | `Pipeline(stages=[...])` |
| **Evaluator** | Mide el rendimiento del modelo | `MulticlassClassificationEvaluator` |

### Feature Engineering

Los modelos de MLlib requieren que las features esten en un **vector numerico**. El proceso tipico es:

1. **StringIndexer:** Convierte columnas categoricas (texto) a indices numericos
2. **Binarizer:** Convierte una columna numerica a binaria (0/1) segun un umbral
3. **VectorAssembler:** Combina multiples columnas en un solo vector de features

### Pipeline ML

Un Pipeline encadena todos los pasos de transformacion y entrenamiento:
```
Datos -> StringIndexer -> Binarizer -> VectorAssembler -> Clasificador -> Predicciones
```
Ventajas:
- Reproducibilidad: mismo pipeline para train y test
- Facilidad: un solo `fit()` entrena todo
- Prevencion de data leakage

## Setup

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.ml.feature import StringIndexer, VectorAssembler, Binarizer
from pyspark.ml.classification import DecisionTreeClassifier, RandomForestClassifier
from pyspark.ml.evaluation import MulticlassClassificationEvaluator, BinaryClassificationEvaluator
from pyspark.ml import Pipeline

spark = SparkSession.builder \
    .appName("ml_spark") \
    .master("local[*]") \
    .getOrCreate()

print(f"Spark version: {spark.version}")

## 1. Cargar y Explorar Datos

Usaremos el dataset de vuelos para predecir si un vuelo tendra retraso. Primero exploramos la distribucion de retrasos.

In [ ]:
# Leer datos de vuelos
df = spark.read.csv("/home/jovyan/datos/flights.csv", header=True, inferSchema=True)

print(f"Total de registros: {df.count()}")
print(f"Columnas: {len(df.columns)}")
df.printSchema()

In [ ]:
# Explorar distribucion de retrasos
df.select("DEPARTURE_DELAY").describe().show()

In [ ]:
# Ver distribucion: vuelos retrasados vs a tiempo (umbral: 15 minutos)
df.groupBy(
    F.when(F.col("DEPARTURE_DELAY") > 15, "Retrasado") \
     .otherwise("A tiempo") \
     .alias("estado")
).count().show()

## 2. Preparar Datos

Antes de entrenar el modelo, necesitamos limpiar los datos eliminando filas con valores nulos en las columnas clave.

In [ ]:
# Eliminar filas con valores nulos en columnas clave
df_clean = df.dropna(subset=[
    "DEPARTURE_DELAY", "AIRLINE", "ORIGIN_AIRPORT", 
    "DESTINATION_AIRPORT", "DISTANCE", "MONTH", "DAY_OF_WEEK"
])

print(f"Registros originales: {df.count()}")
print(f"Registros despues de limpieza: {df_clean.count()}")
print(f"Registros eliminados: {df.count() - df_clean.count()}")

## 3. Feature Engineering: StringIndexer

Los modelos de ML necesitan datos numericos. `StringIndexer` convierte columnas de texto (como AIRLINE) en indices numericos.

In [ ]:
# Codificar variables categoricas con StringIndexer
indexer_airline = StringIndexer(inputCol="AIRLINE", outputCol="airline_idx")
indexer_origin = StringIndexer(inputCol="ORIGIN_AIRPORT", outputCol="origin_idx")

# Ejemplo: ver como funciona StringIndexer
modelo_idx = indexer_airline.fit(df_clean)
df_ejemplo = modelo_idx.transform(df_clean)
df_ejemplo.select("AIRLINE", "airline_idx").distinct().orderBy("airline_idx").show()

## 4. Feature Engineering: Binarizer

`Binarizer` convierte una columna numerica en binaria (0 o 1) segun un umbral. Usaremos un umbral de 15 minutos para definir "retrasado".

In [ ]:
# Crear variable target binaria: 1 si DEPARTURE_DELAY > 15, 0 si no
binarizer = Binarizer(threshold=15.0, inputCol="DEPARTURE_DELAY", outputCol="label")

# Ejemplo: ver el resultado del Binarizer
df_bin = binarizer.transform(df_clean)
df_bin.select("DEPARTURE_DELAY", "label").show(10)

# Distribucion de la variable target
df_bin.groupBy("label").count().show()

## 5. Feature Engineering: VectorAssembler

`VectorAssembler` combina multiples columnas en un unico vector de features, que es el formato requerido por los modelos de MLlib.

In [ ]:
# Combinar features en un solo vector
assembler = VectorAssembler(
    inputCols=["airline_idx", "origin_idx", "MONTH", "DAY_OF_WEEK", "DISTANCE"],
    outputCol="features"
)

print("Features que se usaran:")
print("  - airline_idx: aerolinea codificada")
print("  - origin_idx: aeropuerto origen codificado")
print("  - MONTH: mes del vuelo")
print("  - DAY_OF_WEEK: dia de la semana")
print("  - DISTANCE: distancia del vuelo")

## 6. Division Train/Test

Dividimos los datos en 80% entrenamiento y 20% prueba. Usamos `seed=42` para reproducibilidad.

In [ ]:
# Dividir datos en train y test
train, test = df_clean.randomSplit([0.8, 0.2], seed=42)

print(f"Datos de entrenamiento: {train.count()}")
print(f"Datos de prueba: {test.count()}")

## 7. Modelo: Decision Tree Classifier

El arbol de decision es un modelo interpretable que divide los datos en nodos basandose en las features. `maxDepth` controla la profundidad maxima del arbol.

In [ ]:
# Crear clasificador de arbol de decision
dt = DecisionTreeClassifier(labelCol="label", featuresCol="features", maxDepth=5)

print("Modelo: DecisionTreeClassifier")
print(f"  maxDepth: {dt.getMaxDepth()}")
print(f"  labelCol: {dt.getLabelCol()}")
print(f"  featuresCol: {dt.getFeaturesCol()}")

## 8. Construir y Entrenar Pipeline

El Pipeline encadena todos los pasos: codificacion, binarizacion, ensamblado de features y clasificador. Con un solo `fit()` se entrena todo.

In [ ]:
# Construir pipeline completo
pipeline = Pipeline(stages=[
    indexer_airline,   # Codificar AIRLINE
    indexer_origin,    # Codificar ORIGIN_AIRPORT
    binarizer,         # Crear label binaria
    assembler,         # Combinar features
    dt                 # Clasificador
])

print("Pipeline creado con 5 etapas:")
for i, stage in enumerate(pipeline.getStages()):
    print(f"  {i+1}. {type(stage).__name__}")

In [ ]:
# Entrenar el pipeline
print("Entrenando pipeline...")
model = pipeline.fit(train)
print("Pipeline entrenado exitosamente!")

In [ ]:
# Generar predicciones sobre datos de prueba
predictions = model.transform(test)

# Ver resultados
predictions.select("AIRLINE", "ORIGIN_AIRPORT", "DEPARTURE_DELAY", "label", "prediction", "probability").show(10)

## 9. Evaluacion del Modelo

Usamos `MulticlassClassificationEvaluator` para medir la precision (accuracy) del modelo y `BinaryClassificationEvaluator` para el area bajo la curva ROC.

In [ ]:
# Evaluar accuracy
evaluator = MulticlassClassificationEvaluator(
    labelCol="label", 
    predictionCol="prediction", 
    metricName="accuracy"
)
accuracy = evaluator.evaluate(predictions)
print(f"Accuracy: {accuracy:.4f}")

# Evaluar precision y recall
evaluator_precision = MulticlassClassificationEvaluator(
    labelCol="label", predictionCol="prediction", metricName="weightedPrecision"
)
evaluator_recall = MulticlassClassificationEvaluator(
    labelCol="label", predictionCol="prediction", metricName="weightedRecall"
)
precision = evaluator_precision.evaluate(predictions)
recall = evaluator_recall.evaluate(predictions)

print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")

In [ ]:
# Evaluar AUC (Area Under ROC Curve)
evaluator_auc = BinaryClassificationEvaluator(
    labelCol="label", 
    rawPredictionCol="rawPrediction", 
    metricName="areaUnderROC"
)
auc = evaluator_auc.evaluate(predictions)
print(f"AUC-ROC: {auc:.4f}")

In [ ]:
# Matriz de confusion
predictions.groupBy("label", "prediction").count().orderBy("label", "prediction").show()

In [ ]:
# Importancia de features (solo para arboles de decision)
dt_model = model.stages[-1]  # Ultimo stage es el clasificador entrenado
feature_names = ["airline_idx", "origin_idx", "MONTH", "DAY_OF_WEEK", "DISTANCE"]

print("Importancia de Features:")
print("-" * 35)
for name, importance in zip(feature_names, dt_model.featureImportances.toArray()):
    print(f"  {name:20s}: {importance:.4f}")

---
## Ejercicios

Ahora es tu turno de practicar. Completa los siguientes ejercicios.

In [ ]:
# =============================================================
# EJERCICIO 1: Agregar mas features al modelo
# =============================================================
# Mejorar el modelo agregando DESTINATION_AIRPORT como feature

# 1. Crear StringIndexer para DESTINATION_AIRPORT
indexer_dest = StringIndexer(inputCol="DESTINATION_AIRPORT", outputCol="dest_idx")

# 2. Crear nuevo VectorAssembler con dest_idx incluido
assembler_v2 = VectorAssembler(
    inputCols=["airline_idx", "origin_idx", "dest_idx", "MONTH", "DAY_OF_WEEK", "DISTANCE"],
    outputCol="features"
)

# 3. Crear clasificador
dt_v2 = DecisionTreeClassifier(labelCol="label", featuresCol="features", maxDepth=5)

# 4. Crear nuevo pipeline con 6 stages
pipeline_v2 = Pipeline(stages=[
    indexer_airline,    # Codificar AIRLINE
    indexer_origin,     # Codificar ORIGIN_AIRPORT
    indexer_dest,       # Codificar DESTINATION_AIRPORT (nuevo)
    binarizer,          # Crear label binaria
    assembler_v2,       # Combinar features (incluye dest_idx)
    dt_v2               # Clasificador
])

print("Pipeline v2 creado con 6 etapas:")
for i, stage in enumerate(pipeline_v2.getStages()):
    print(f"  {i+1}. {type(stage).__name__}")

# 5. Entrenar y evaluar
print("\nEntrenando pipeline v2...")
model_v2 = pipeline_v2.fit(train)
predictions_v2 = model_v2.transform(test)

# Evaluar
accuracy_v2 = evaluator.evaluate(predictions_v2)
auc_v2 = evaluator_auc.evaluate(predictions_v2)

# 6. Comparar con modelo anterior
print(f"\n{'Metrica':<20s} {'Modelo v1':>12s} {'Modelo v2':>12s} {'Diferencia':>12s}")
print("-" * 58)
print(f"{'Accuracy':<20s} {accuracy:>12.4f} {accuracy_v2:>12.4f} {accuracy_v2 - accuracy:>+12.4f}")
print(f"{'AUC-ROC':<20s} {auc:>12.4f} {auc_v2:>12.4f} {auc_v2 - auc:>+12.4f}")

# Importancia de features del modelo v2
dt_model_v2 = model_v2.stages[-1]
feature_names_v2 = ["airline_idx", "origin_idx", "dest_idx", "MONTH", "DAY_OF_WEEK", "DISTANCE"]

print("\nImportancia de Features (v2):")
print("-" * 35)
for name, importance in zip(feature_names_v2, dt_model_v2.featureImportances.toArray()):
    print(f"  {name:20s}: {importance:.4f}")

> **Nota docente:** Agregar DESTINATION_AIRPORT como feature puede mejorar o no
> el modelo. Puntos clave:
>
> - Al agregar mas features, el modelo tiene mas informacion para aprender, pero
>   tambien puede sobreajustarse (overfitting) si las features no son relevantes.
> - `StringIndexer` asigna indices por frecuencia: el aeropuerto mas comun recibe
>   indice 0. Esto introduce un sesgo ordinal que modelos como arboles pueden manejar,
>   pero para modelos lineales se necesitaria `OneHotEncoder`.
> - Con `handleInvalid="keep"` en StringIndexer se pueden manejar categorias nuevas
>   en test que no estaban en train.
>
> Error comun: agregar features altamente correlacionadas (como ORIGIN y DESTINATION
> que juntas determinan DISTANCE), lo cual no mejora el modelo pero si aumenta el
> costo computacional.

In [ ]:
# =============================================================
# EJERCICIO 2: Random Forest Classifier
# =============================================================
# Reemplazar DecisionTreeClassifier por RandomForestClassifier

# 1. Crear RandomForestClassifier
rf = RandomForestClassifier(
    labelCol="label", 
    featuresCol="features", 
    numTrees=20
)

print(f"Modelo: RandomForestClassifier")
print(f"  numTrees: {rf.getNumTrees()}")
print(f"  maxDepth: {rf.getMaxDepth()}")

# 2. Crear pipeline con RandomForest (usando las mismas features del modelo base)
pipeline_rf = Pipeline(stages=[
    indexer_airline,   # Codificar AIRLINE
    indexer_origin,    # Codificar ORIGIN_AIRPORT
    binarizer,         # Crear label binaria
    assembler,         # Combinar features
    rf                 # RandomForest (en lugar de DecisionTree)
])

# 3. Entrenar y predecir
print("\nEntrenando RandomForest...")
model_rf = pipeline_rf.fit(train)
predictions_rf = model_rf.transform(test)
print("Entrenamiento completado!")

# 4. Evaluar
accuracy_rf = evaluator.evaluate(predictions_rf)
precision_rf = evaluator_precision.evaluate(predictions_rf)
recall_rf = evaluator_recall.evaluate(predictions_rf)
auc_rf = evaluator_auc.evaluate(predictions_rf)

print(f"\nResultados RandomForest:")
print(f"  Accuracy:  {accuracy_rf:.4f}")
print(f"  Precision: {precision_rf:.4f}")
print(f"  Recall:    {recall_rf:.4f}")
print(f"  AUC-ROC:   {auc_rf:.4f}")

# 5. Comparar con DecisionTree
print(f"\n{'Metrica':<20s} {'DecisionTree':>14s} {'RandomForest':>14s} {'Diferencia':>12s}")
print("-" * 62)
print(f"{'Accuracy':<20s} {accuracy:>14.4f} {accuracy_rf:>14.4f} {accuracy_rf - accuracy:>+12.4f}")
print(f"{'AUC-ROC':<20s} {auc:>14.4f} {auc_rf:>14.4f} {auc_rf - auc:>+12.4f}")

if accuracy_rf > accuracy:
    print("\n=> RandomForest tiene mejor accuracy")
else:
    print("\n=> DecisionTree tiene mejor accuracy")

if auc_rf > auc:
    print("=> RandomForest tiene mejor AUC")
else:
    print("=> DecisionTree tiene mejor AUC")

# Matriz de confusion
print("\nMatriz de confusion (RandomForest):")
predictions_rf.groupBy("label", "prediction").count().orderBy("label", "prediction").show()

> **Nota docente:** RandomForest es un **ensemble** de multiples arboles de decision.
> Generalmente produce mejores resultados que un arbol individual porque:
> - Cada arbol se entrena con una muestra aleatoria de los datos (bagging)
> - En cada nodo, solo considera un subconjunto aleatorio de features
> - La prediccion final es el voto mayoritario de todos los arboles
>
> Esto reduce el overfitting significativamente. El parametro clave es `numTrees`:
> mas arboles = mejor precision pero mayor costo computacional.
>
> En este caso, con `numTrees=20` se obtiene un balance razonable. En produccion,
> valores de 100-500 arboles son comunes.
>
> Error comun: comparar modelos con diferentes conjuntos de features. Aqui usamos
> las mismas features para que la comparacion sea justa.

---
## Desafio Extra (Opcional)

**Para estudiantes avanzados:**

Busqueda de hiperparametros con CrossValidator.

In [ ]:
# =============================================================
# DESAFIO: CrossValidator para busqueda de hiperparametros
# =============================================================

from pyspark.ml.tuning import CrossValidator, ParamGridBuilder

# 1. Preparar datos con el pipeline base (sin clasificador)
pipeline_base = Pipeline(stages=[
    indexer_airline, 
    indexer_origin, 
    binarizer, 
    assembler
])

# Entrenar pipeline base y transformar datos
pipeline_base_model = pipeline_base.fit(train)
df_train_prepared = pipeline_base_model.transform(train)
df_test_prepared = pipeline_base_model.transform(test)

print(f"Datos preparados - Train: {df_train_prepared.count()}, Test: {df_test_prepared.count()}")

# 2. Crear RandomForestClassifier
rf_cv = RandomForestClassifier(labelCol="label", featuresCol="features")

# 3. Crear grid de parametros
paramGrid = ParamGridBuilder() \
    .addGrid(rf_cv.maxDepth, [3, 5, 7, 10]) \
    .addGrid(rf_cv.numTrees, [10, 20, 50]) \
    .build()

print(f"Combinaciones de hiperparametros a probar: {len(paramGrid)}")
print("  maxDepth: [3, 5, 7, 10]")
print("  numTrees: [10, 20, 50]")
print(f"  Total combinaciones: 4 x 3 = {len(paramGrid)}")

# 4. Crear CrossValidator
cv = CrossValidator(
    estimator=rf_cv,
    estimatorParamMaps=paramGrid,
    evaluator=BinaryClassificationEvaluator(labelCol="label"),
    numFolds=3
)

print(f"\nCross-validation con {cv.getNumFolds()} folds")
print(f"Total de modelos a entrenar: {len(paramGrid)} x {cv.getNumFolds()} = {len(paramGrid) * cv.getNumFolds()}")

# 5. Entrenar (NOTA: puede tardar varios minutos)
print("\nEntrenando CrossValidator (esto puede tardar unos minutos)...")
cvModel = cv.fit(df_train_prepared)
print("CrossValidator entrenado exitosamente!")

In [ ]:
# 6. Obtener el mejor modelo
best_model = cvModel.bestModel

print("=== Mejor modelo encontrado ===")
print(f"  maxDepth: {best_model.getMaxDepth()}")
print(f"  numTrees: {best_model.getNumTrees}")

# 7. Evaluar el mejor modelo en test
predictions_cv = cvModel.transform(df_test_prepared)

accuracy_cv = evaluator.evaluate(predictions_cv)
auc_cv = evaluator_auc.evaluate(predictions_cv)

print(f"\n=== Metricas del mejor modelo ===")
print(f"  Accuracy: {accuracy_cv:.4f}")
print(f"  AUC-ROC:  {auc_cv:.4f}")

# 8. Comparar todos los modelos
print(f"\n{'Modelo':<25s} {'Accuracy':>10s} {'AUC':>10s}")
print("-" * 47)
print(f"{'DecisionTree (base)':<25s} {accuracy:>10.4f} {auc:>10.4f}")
print(f"{'RandomForest (20 trees)':<25s} {accuracy_rf:>10.4f} {auc_rf:>10.4f}")
print(f"{'CrossValidator (mejor)':<25s} {accuracy_cv:>10.4f} {auc_cv:>10.4f}")

# 9. Mostrar resultados de validacion cruzada por combinacion
print("\n=== Resultados por combinacion (AUC promedio) ===")
avg_metrics = cvModel.avgMetrics
for i, (params, metric) in enumerate(zip(paramGrid, avg_metrics)):
    depth = params[rf_cv.maxDepth]
    trees = params[rf_cv.numTrees]
    marker = " <-- MEJOR" if metric == max(avg_metrics) else ""
    print(f"  maxDepth={depth:>2d}, numTrees={trees:>2d} => AUC={metric:.4f}{marker}")

> **Nota docente:** CrossValidator es la herramienta estandar para busqueda de
> hiperparametros en Spark MLlib. Puntos clave:
>
> - **ParamGridBuilder** define una grilla de combinaciones. Cada combinacion se
>   evalua con validacion cruzada de k-folds.
> - **numFolds=3** significa que cada combinacion se entrena 3 veces con diferentes
>   splits. Total de modelos = combinaciones x folds = 12 x 3 = 36.
> - El costo computacional crece rapidamente con mas parametros y mas folds.
>   En produccion se usa `TrainValidationSplit` (1 split, mas rapido) para
>   busquedas iniciales y CrossValidator para ajuste fino.
>
> Separacion en 2 pasos (pipeline_base + clasificador):
> - Se preparan los datos una sola vez con pipeline_base
> - Solo se hace CV sobre el clasificador, evitando re-ejecutar StringIndexer etc.
>   36 veces. Esto reduce significativamente el tiempo de ejecucion.
>
> Error comun: incluir todo el pipeline dentro de CrossValidator. Funciona pero
> es mucho mas lento porque repite las transformaciones en cada fold.

---
## Resumen

En esta actividad aprendimos:

1. **MLlib:** Libreria de Machine Learning distribuida de Spark
2. **StringIndexer:** Codificar variables categoricas a indices numericos
3. **Binarizer:** Crear variable target binaria a partir de un umbral
4. **VectorAssembler:** Combinar multiples columnas en un vector de features
5. **Pipeline:** Encadenar transformaciones y modelo en un flujo reproducible
6. **DecisionTreeClassifier:** Modelo de arbol de decision para clasificacion
7. **RandomForestClassifier:** Ensemble de arboles para mejor precision
8. **Evaluadores:** Accuracy, precision, recall y AUC para medir rendimiento
9. **CrossValidator:** Busqueda sistematica de hiperparametros optimos

In [ ]:
# Detener la SparkSession al finalizar
spark.stop()
print("SparkSession detenida correctamente.")